# 02 · Posterior summaries over the sample axis

**What this teaches:** given a `PredictionFrame` whose rows are *samples* from an approximate predictive posterior, what summaries can `views_frames_summarize` read off the draws — point estimate, credible intervals, exceedance probabilities, a robust worst-case, and a multimodality flag — and **how each behaves across a zoo of distribution shapes** (the heart of the notebook).

**Audience:** anyone choosing which summary to publish, and wanting to *see* why the nonparametric, read-off-the-draws approach behaves well on ugly real posteriors.

> **🚧 ROADMAP — draft, not yet built.** This notebook currently holds only the *plan* (markdown). The section outline below is what we align on **before** the first content pass; code cells get filled in afterwards. Edit these markdown cells freely — this is the working document for the notebook's design.

## 📋 Roadmap (section-by-section)

**1. The estimand & the philosophy.** Real conflict posteriors are low-sample, zero-inflated, right-skewed, heavy-tailed, occasionally bimodal. The approach: read summaries *off the empirical draws*, one weak assumption (approx. unimodal), flag where it fails. No parametric fit.

**2. The distribution zoo (the spine of the notebook).** A synthetic battery of shapes — zero-inflated, right-skewed (gamma/lognormal), near-symmetric, heavy-tailed, genuinely bimodal, low-sample (S≈32) vs pooled (S≈1024). Plot the histograms in a grid. *Reuse `research/map_hdi/benchmark/battery.py`.* Every later section sweeps over this zoo.

**3. Point estimate.** `map_estimate` (histogram MAP — show the C-32 lowest-index downward bias on skewed/zero-inflated) vs `tower_point` (the shorth tip — robust). Show the disagreement explicitly; this is the C-32 story.

**4. The HDI tower.** `hdi_tower` — nested-by-construction, reproducible bands (50/90/95/99). The **tower-overlay plot** across the zoo (reuse the `tower_overlay` / `tower_detail` figures from the FAO note as the visual template). Show nesting + the MAP-inside-all-bands property.

**5. Quantiles & exceedance.** `quantiles`; `exceedance` — `P(Y > c)`, the onset `P(Y > 0)` flagship, a threshold sweep → a survival curve per shape. Strict `>`, fail-loud on non-finite.

**6. Worst-case severity.** `expected_shortfall` (mean of the worst `⌈t·S⌉` draws — tail mean / CVaR) over the zoo; the **small-tail → max degeneracy caveat** (needs enough draws); vs the volatile raw `max`. Tie to the FAO `severe_scenario`.

**7. Multimodality flag.** `bimodality` — fires on the bimodal shapes, *stays quiet* on skewed/zero-inflated (the conservative tuning). The safeguard story.

**8. Cross-level aggregation.** `aggregate_distributions` (joint-sample element-wise sum pgm→cm); demonstrate `HDI(sum) ≠ sum(HDI)` and why joint sampling matters.

**9. Laws as a coda.** `assert_summarizer_contract` — the properties (nesting, reproducibility, `ES ≥ (1−t) quantile`, in-[0,1] exceedance) hold by construction.

## 🔬 Additions from the expert-method-review panel (2026-06-26)

*Folded in from the practitioner panel (Gelman · Gneiting · Hyndman · Wilke/Kay · VIEWS-FAO). These slot into the section-by-section roadmap above. Tracked as register C-59/C-61.*

- **NEW §A — Calibration & coverage (highest priority; C-59).** On synthetic draws with known truth, show empirical **coverage** of each HDI/quantile band (does the 90% band cover ~90%?) and a **PIT histogram** (flat ⇒ calibrated). A notebook that displays named intervals must demonstrate they hold. *(Gneiting2014; Kuleshov2018.)* Note: this is a property demo on synthetic truth, **not** scoring — scoring stays in views-evaluation.
- **NEW §B — Equal-tailed vs HDI, and mean/median beside the MAP.** Show the equal-tailed quantile interval next to the HDI with the honest tradeoff (**HDI is sharp but not invariant** under monotone reparameterization; ET is invariant); show mean & median alongside the tower-tip and state plainly that **the mode is a contested estimand**. *(Gelman / BDA3.)*
- **NEW §C — Sample-size sensitivity / Monte-Carlo error.** Each summary's sampling variability at **S≈32 vs S≈1024** (repeat the draw, show error bands) — the estimand's defining axis.
- **NEW §D — "When each summary misleads."** The failure modes, shown not just stated: small-S `expected_shortfall` → `max`; the bimodal cell where any single point is meaningless; the HDI splitting across modes; `MAP = 0` correct on a zero-inflated cell where a naive mean misleads. *(All seats — honesty about failure earns trust.)*
- **NEW §E — Known-truth recovery (PPC-flavored).** Overlay the true generating value; does the summary recover it across the zoo? *(Gelman.)*
- **NEW §F — Spatial / map view on a toy lattice (C-61).** A synthetic square grid of cells → a choropleth of MAP / `P(Y>0)` / `severe_scenario`. The most operationally-expected view for a spatial-forecasting audience; uses a **toy lattice only** (no domain geography — ADR-014). *(Wilke/Kay + domain.)*
- **NEW §G — Decision relevance.** Tie onset `exceedance(frame, [0])` and `severe_scenario` to an FAO-style threshold/alert ("alert if P(Y>0) > τ; plan to the severe_scenario"). *(Bernardo1979 — expected utility; the summaries serve a decision.)*

## 🎨 Source material — self-contained (logic ported in, nothing referenced externally)

This notebook **regenerates its own data and plots from code**; it depends on no file outside
this repo.

- *In-repo to adapt:* `research/map_hdi/` — the synthetic distribution battery (`benchmark/battery.py`, with known analytic modes) and the tower/density plotting (`audit_tower.py`, `density_sweep.py`, `point_pass.py`).
- The tower-overlay / tower-detail visuals are **re-created in-notebook** from the same construction — not embedded from any external PDF.
- Earlier HDI/tower exploration exists as prior prototypes; during the build we **port the useful plotting logic into this notebook** (copied in, self-contained). We do not reference files scattered outside the repo.

## ❓ Open questions / decisions

1. **Static plots vs interactive** — an `ipywidgets` slider that sweeps the zoo / the tail fraction / the credible mass, or a fixed grid of stills? (Interactive is compelling but adds a dep + doesn't render on GitHub.)
2. The **shrinking-HDI → MAP animation** (the α→0 limit, issue #89) — include as a teaser, or out of scope?
3. Exactly **which shapes** make the zoo (and how many) — and S values (32 vs 1024 to show the small-sample effects).
4. Regenerate the tower figures in-notebook (self-contained) vs embed the existing FAO PDFs (faster, but external).

## ✅ Conventions for these notebooks

- **Public, frozen API only.** Everything shown uses the published `views_frames` / `views_frames_summarize` / `views_frames_reconcile` surface — so the notebook doubles as a living contract demo. If a cell *wants* a convenience that doesn't exist (e.g. `frame.to_parquet`), that's a **demand signal** to record, not a reason to reach into the leaf (see register D-11).
- **Synthetic data only.** No `viewser`, no domain fetches, no `views_*` consumer imports — every array is generated in-notebook with a fixed seed. Zero domain knowledge (ADR-001).
- **Un-gated dev artifact.** Lives in `notebooks/` (like `research/` and `scripts/`), outside `src/`; the package never imports it, so the import-DAG and the frozen core are untouched. Plotting/jupyter deps are an **optional extra**, never runtime deps.
- **Reproducible + light.** Fixed seeds; small sample sizes that render fast; no heavyweight state. A reader should be able to *Run All* in well under a minute.